<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# E-Commerce Analytics 360
## Étape 2 — Data Cleaning & Feature Engineering
### ✅ VERSION CORRIGÉE

> **Comment lire ce corrigé :**  
> Les blocs `MÉTHODE` expliquent les choix techniques et les bonnes pratiques.  
> Les blocs `INTERPRÉTATION` lisent les résultats comme le ferait un analyste en restitution.  
> Les blocs `MÉTIER` font le lien entre le chiffre et la décision business.

| | |
|---|---|
| **Niveau** | Intermédiaire / Avancé |
| **Outils** | Python — pandas, numpy, matplotlib, sqlalchemy |
| **Durée estimée** | 2h à 3h |

---
## 0. Mise en place de l'environnement

> **MÉTHODE — Pourquoi reconfigurer l'environnement dans chaque notebook ?**
>
> Chaque notebook doit être **autonome** et ré-exécutable indépendamment. Si on dépend de variables définies dans l'étape 1, le notebook ne fonctionne plus ouvert seul ou dans un pipeline automatisé.
>
> `pd.set_option('display.float_format', '{:.2f}'.format)` évite l'affichage de `7189012.73000000001` au lieu de `7189012.73`. En finance et en e-commerce, les arrondis mal affichés créent de la défiance lors des restitutions.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings


import os
import sys

# Détecter si on est dans Colab ou en local
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/projects/ecommerce_analytics/"
else:
    SAVE_PATH = "./outputs/"

# Chemin pour enregistrer les fichiers exportés
os.makedirs(SAVE_PATH, exist_ok=True)
print(f" Environnement Colab : {'Colab' if IN_COLAB else 'Local'}")
print(f" Dossier de travail : {SAVE_PATH}")

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:.2f}".format)

COLORS = {
    "primary":   "#534AB7",
    "secondary": "#1D9E75",
    "warning":   "#EF9F27",
    "danger":    "#E24B4A",
    "neutral":   "#888780",
}

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F9F9F8",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "font.size":        11,
})

print("Environnement prêt ✅")

---
## 1. Chargement des données

> **MÉTHODE — On recharge depuis les CSV bruts, pas depuis l'étape 1.**
>
> Règle de base : chaque notebook recharge ses propres données. Si on partait des variables de l'étape 1, ce notebook serait inutilisable ouvert seul ou dans un pipeline automatisé.
>
> **`parse_dates` est obligatoire sur toutes les colonnes temporelles.** Sans lui, `order_date` est un `object` (string) — tri alphabétique, agrégations temporelles impossibles.

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/data/"

customers    = pd.read_csv(BASE_URL + "customers.csv",    parse_dates=["date_inscription"])
products     = pd.read_csv(BASE_URL + "products.csv")
orders       = pd.read_csv(BASE_URL + "orders.csv",       parse_dates=["order_date"])
order_items  = pd.read_csv(BASE_URL + "order_items.csv")
reviews      = pd.read_csv(BASE_URL + "reviews.csv")
web_logs     = pd.read_csv(BASE_URL + "web_logs.csv",     parse_dates=["timestamp"])

print(f"{'Table':<15} {'Lignes':>8} {'Colonnes':>10}")
print("-" * 36)
for nm, df in [("customers", customers), ("products", products),
               ("orders", orders), ("order_items", order_items)]:
    print(f"{nm:<15} {len(df):>8,} {len(df.columns):>10}")

---
## 2. Audit qualité

> **MÉTHODE — Pourquoi auditer avant de nettoyer ?**
>
> L'audit détermine QUOI nettoyer et COMMENT. Nettoyer sans audit revient à réparer une voiture sans avoir diagnostiqué la panne. L'audit se déroule en 4 étapes dans l'ordre : types → nulls → doublons → cohérence. Chaque anomalie génère une **décision de traitement** documentée.

### 2.1 — Types de données

> **MÉTHODE — Ce qu'on vérifie sur les types.**
>
> - Les colonnes de dates sont-elles en `datetime64` ?
> - Les colonnes numériques sont-elles en `float64` ou `int64` ? (pas `object`)
>
> **Un type `object` sur une colonne numérique** signifie qu'il y a des valeurs non-numériques cachées (espaces, virgules comme séparateur décimal) qui ont forcé pandas à choisir le type le plus général.

In [ ]:
cols_texte = {"customer_id", "product_id", "order_id", "pays", "ville",
              "segment_client", "categorie", "marque", "canal",
              "payment_method", "order_status", "nom_produit"}

for nom, df in [("customers", customers), ("products", products),
                ("orders", orders), ("order_items", order_items)]:
    print(f"\n{'='*45}")
    print(f"  {nom.upper()}  —  {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
    print(f"{'='*45}")
    for col, dtype in df.dtypes.items():
        flag = "  ← VÉRIFIER : attendu numérique ?" if dtype == object and col not in cols_texte else ""
        print(f"  {col:<30} : {str(dtype):<12}{flag}")

### 2.2 — Valeurs manquantes

> **MÉTHODE — Trois catégories de nulls, trois traitements différents.**
>
> **Null légitime** : la valeur n'est pas applicable dans ce contexte. Ex : pas de `delivery_delay` sur une commande annulée → ne pas imputer.
>
> **Null technique** : la valeur devrait être présente mais manque. Ex : `delivery_delay` null sur une commande livrée → imputer par la médiane.
>
> **Null critique** : la valeur manque sur une colonne indispensable. Ex : `montant_total` null → exclure la ligne.
>
> **Pourquoi la médiane et pas la moyenne pour `delivery_delay` ?** Les délais ont une distribution asymétrique à droite (quelques colis perdus tirent la moyenne vers le haut). La médiane représente le cas typique.

In [ ]:
print("RAPPORT VALEURS MANQUANTES")
print("=" * 60)

rapport = []
for nm, df in [("customers", customers), ("products", products),
               ("orders", orders), ("order_items", order_items)]:
    total_nulls = df.isnull().sum().sum()
    rapport.append({"dataset": nm, "lignes": len(df), "colonnes": len(df.columns),
                    "total_nulls": total_nulls,
                    "pct_cellules_nulles": round(total_nulls / df.size * 100, 2)})

display(pd.DataFrame(rapport))

print("\nDétail par colonne (colonnes avec nulls uniquement) :")
for nm, df in [("customers", customers), ("products", products),
               ("orders", orders), ("order_items", order_items)]:
    nulls = df.isnull().sum()
    nulls_nz = nulls[nulls > 0]
    if len(nulls_nz) > 0:
        print(f"\n  {nm.upper()} :")
        for col, n in nulls_nz.items():
            pct = n / len(df) * 100
            print(f"    {col:<30} : {n:>5} nulls ({pct:.1f}%)")

> **INTERPRÉTATION — Analyse des valeurs manquantes**
>
> **`segment_client` dans customers :** Les nouveaux clients sans historique n'ont pas encore été segmentés. Ce null est semi-légitime → imputer par `'Standard'` (segment neutre par défaut).
>
> **`delivery_delay` null sur commandes livrées :** Erreur de saisie → imputer par la médiane. Sur commandes en cours : légitime, laisser null.
>
> **`montant_total` null (si détecté) — CRITIQUE :** Exclure ces lignes. Si > 1% des commandes, investiguer avant d'exclure.

### 2.3 — Doublons

> **MÉTHODE — Deux types de doublons, deux stratégies différentes.**
>
> **Doublon de clé primaire** : deux commandes avec le même `order_id`. Cause probable : double export, erreur ETL. Stratégie : `keep='first'`.
>
> **Doublon de ligne complète** : deux lignes identiques sur toutes les colonnes. Cause probable : import double du même fichier. Stratégie : suppression directe.

In [ ]:
print("AUDIT DES DOUBLONS")
print("-" * 55)

for nm, df, pk in [
    ("customers",   customers,   "customer_id"),
    ("products",    products,    "product_id"),
    ("orders",      orders,      "order_id"),
    ("order_items", order_items, None),
]:
    dup_full = df.duplicated().sum()
    dup_pk   = df.duplicated(subset=[pk]).sum() if pk else "N/A"
    status   = "✅ OK" if dup_full == 0 else "⚠️ ATTENTION"
    print(f"  [{status}] {nm:<15} | Doublons ligne : {dup_full:>4} | Doublons PK : {str(dup_pk):>4}")

### 2.4 — Cohérence globale

> **MÉTHODE — Les 4 vérifications de cohérence indispensables en e-commerce.**
>
> 1. **Intégrité référentielle** : les FK pointent-elles vers des enregistrements existants ?
> 2. **Montants négatifs** : un `montant_total < 0` est une erreur de saisie si les remboursements sont gérés via `order_status`.
> 3. **Délais négatifs** : `delivery_delay < 0` est physiquement impossible.
> 4. **Valeurs de statut** : les variantes `'Livree '` (espace), `'livree'` (minuscule) brisent tous les filtres.

In [ ]:
print("COHÉRENCE GLOBALE")
print("-" * 55)

orphans_clients  = orders[~orders["customer_id"].isin(customers["customer_id"])]
orphans_products = order_items[~order_items["product_id"].isin(products["product_id"])]
neg_mt   = orders[orders["montant_total"] < 0]
neg_dd   = orders[orders["delivery_delay"] < 0]

statuts_valides = {"Livree", "En cours", "Annulee", "Remboursee"}
invalides = orders[~orders["order_status"].isin(statuts_valides)]
livrees_sans_delai = orders[(orders["order_status"] == "Livree") & (orders["delivery_delay"].isnull())]

print(f"  FK clients invalides  : {len(orphans_clients):>4}")
print(f"  FK produits invalides : {len(orphans_products):>4}")
print(f"  Montants négatifs     : {len(neg_mt):>4}")
print(f"  Délais négatifs       : {len(neg_dd):>4}")
print(f"  Statuts invalides     : {len(invalides):>4}")
print(f"  Livrées sans délai    : {len(livrees_sans_delai):>4} (anomalie technique)")
print(f"\n  Statuts présents : {sorted(orders['order_status'].unique().tolist())}")
print(f"  Canaux présents  : {sorted(orders['canal'].unique().tolist())}")

---
## 3. Nettoyage des données

> **MÉTHODE — La règle d'or du nettoyage : toujours documenter la décision.**
>
> Pour chaque opération : **Quoi** (quelle anomalie), **Pourquoi** (quel problème si on ne le fait pas), **Comment** (quelle stratégie et pourquoi).

### 3.1 — Traitement des valeurs manquantes

> **MÉTHODE — `fillna` vs `.mask().fillna()` : la différence.**
>
> `.mask(condition, np.nan).fillna(mediane)` : remplace d'abord les valeurs qui violent une condition par NaN, puis impute. Utile pour convertir des anomalies (délais négatifs) en NaN avant imputation.
>
> **La médiane via `.median()` est calculée sur les données saines.** Si on calcule après avoir mis les négatifs en NaN, pandas ignore automatiquement les NaN dans `.median()`. L'ordre des opérations : `mask` → `median` → `fillna`.

In [ ]:
# Décision 1 : segment_client NULL → "Standard"
# Justification : clients non classés = nouveaux comptes sans historique
# "Standard" est le segment neutre par défaut
n_null_seg = customers["segment_client"].isnull().sum()
customers["segment_client"] = customers["segment_client"].fillna("Standard")
print(f"segment_client : {n_null_seg} nulls imputés par 'Standard'")

# Décision 2 : delivery_delay NULL ou négatif → médiane des valeurs saines
# Justification : médiane car distribution asymétrique (colis perdus = outliers)
orders["delivery_delay"] = orders["delivery_delay"].mask(orders["delivery_delay"] < 0, np.nan)
median_delay = orders["delivery_delay"].median()
n_null_delay = orders["delivery_delay"].isnull().sum()
orders["delivery_delay"] = orders["delivery_delay"].fillna(median_delay)
print(f"delivery_delay : {n_null_delay} nulls imputés par la médiane ({median_delay:.1f} jours)")

# Décision 3 : montant_total NULL → exclusion
# Justification : une commande sans montant est inutilisable pour tout KPI financier
n_null_mt = orders["montant_total"].isnull().sum()
if n_null_mt > 0:
    orders = orders[orders["montant_total"].notna()].copy()
    print(f"montant_total  : {n_null_mt} lignes exclues (données corrompues)")
else:
    print("montant_total  : 0 null — aucune exclusion nécessaire")

print(f"\nVérification post-imputation :")
print(f"  segment_client nulls restants : {customers['segment_client'].isnull().sum()}")
print(f"  delivery_delay nulls restants : {orders['delivery_delay'].isnull().sum()}")

### 3.2 — Déduplication

> **MÉTHODE — `drop_duplicates(subset=cle)` vs `drop_duplicates()` sans argument.**
>
> `drop_duplicates(subset='order_id')` : supprime les lignes où `order_id` est identique à une ligne précédente. Les autres colonnes peuvent différer.
>
> `.reset_index(drop=True)` : recalcule les index après suppression. Sans lui, les index ne sont plus consécutifs (0,1,3,5...) ce qui peut causer des comportements inattendus lors des joins et slices.

In [ ]:
n_before = {
    "customers":   len(customers),
    "orders":      len(orders),
    "order_items": len(order_items),
    "products":    len(products),
}

customers   = customers.drop_duplicates(subset="customer_id",  keep="first").reset_index(drop=True)
orders      = orders.drop_duplicates(subset="order_id",        keep="first").reset_index(drop=True)
products    = products.drop_duplicates(subset="product_id",    keep="first").reset_index(drop=True)
order_items = order_items.drop_duplicates().reset_index(drop=True)

print(f"{'Table':<15} {'Avant':>8} {'Après':>8} {'Supprimé':>10}")
print("-" * 45)
for nm, n in n_before.items():
    df_now = {"customers": customers, "orders": orders,
              "order_items": order_items, "products": products}[nm]
    print(f"{nm:<15} {n:>8,} {len(df_now):>8,} {n-len(df_now):>10,}")

### 3.3 — Anomalies métier

> **MÉTHODE — Distinction anomalie technique vs anomalie métier.**
>
> **Anomalie technique** : valeur impossible physiquement (délai négatif, montant négatif) → exclusion ou imputation.
>
> **Anomalie métier** : valeur possible mais incongrue dans le contexte (panier à 50 000€ sur un site à 100€ max) → investigation avant exclusion. Pour ce projet, on traite les anomalies techniques uniquement.

In [ ]:
# Exclusion des montants négatifs (anomalie technique)
n_neg_mt = (orders["montant_total"] < 0).sum()
orders = orders[orders["montant_total"] >= 0].copy()
print(f"Montants négatifs exclus : {n_neg_mt}")

# Normalisation des statuts (espaces, casse)
orders["order_status"] = orders["order_status"].astype(str).str.strip()
orders["canal"]        = orders["canal"].astype(str).str.strip()
print(f"Statuts après normalisation : {sorted(orders['order_status'].unique())}")
print(f"Canaux après normalisation  : {sorted(orders['canal'].unique())}")
print(f"\nOrders après nettoyage complet : {len(orders):,} lignes")

---
## 4. Feature Engineering

> **MÉTHODE — Le feature engineering précède le ML et facilite l'analyse.**
>
> On crée ici 3 types de variables :
> - **Variables financières** : `revenue`, `cost`, `margin` au niveau ligne.
> - **Agrégats client** : KPIs au niveau client (base de la segmentation RFM en étape 3).
> - **Flags métier** : variables binaires 0/1 qui encodent des règles business.
>
> **Règle d'ordre fondamentale :** variable cible → features → split → entraînement. Ne jamais créer des features à partir de données non disponibles au moment de la prédiction (data leakage).

### 4.1 — Merge principal

> **MÉTHODE — Partir de `order_items` comme table centrale.**
>
> On part de la table à la granularité la plus fine (1 ligne = 1 produit dans 1 commande) et on enrichit progressivement.
>
> **L'`assert` comme filet de sécurité :** si le résultat a plus de lignes qu'`order_items`, c'est qu'`orders` a des doublons non supprimés. Si moins, ce sont des orphelins. L'assert lève une exception immédiatement — technique de programmation défensive essentielle en data engineering.

In [ ]:
df = (
    order_items
    .merge(
        orders[["order_id", "customer_id", "order_date", "canal", "order_status",
                "payment_method", "montant_total", "delivery_delay"]],
        on="order_id", how="inner"
    )
    .merge(
        products[["product_id", "nom_produit", "categorie", "marque", "cout_produit"]],
        on="product_id", how="left"
    )
    .merge(
        customers[["customer_id", "pays", "ville", "segment_client", "date_inscription"]],
        on="customer_id", how="left"
    )
)

assert len(df) == len(order_items), f"ALERTE : {len(df)} lignes vs {len(order_items)} attendues — vérifier les FK"
print(f"Dataset consolidé  : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print("Intégrité du merge : ✅ OK (même nombre de lignes qu'order_items)")

### 4.2 — Variables financières

> **MÉTHODE — Calculer `revenue` et `margin` au niveau ligne (pas ordre).**
>
> `revenue = quantite * prix_unitaire` : prix vient de `order_items` (prix réel payé), pas de `products` (prix catalogue). Un client ayant bénéficié d'une promotion a payé moins que le catalogue — notre calcul le reflète.
>
> **`revenue` total ≠ `montant_total` de `orders` :** Les frais de livraison peuvent être inclus dans `montant_total` mais pas dans `order_items`. Ce n'est pas une erreur — c'est une différence de périmètre à documenter.

In [ ]:
df["revenue"] = df["quantite"] * df["prix_unitaire"]   # CA ligne
df["cost"]    = df["quantite"] * df["cout_produit"]    # Coût ligne
df["margin"]  = df["revenue"] - df["cost"]             # Marge brute ligne
df["mois"]    = df["order_date"].dt.to_period("M")     # Période mensuelle

print("Variables financières créées :")
print(f"  revenue total  : {df['revenue'].sum():>12,.0f} €")
print(f"  cost total     : {df['cost'].sum():>12,.0f} €")
print(f"  margin total   : {df['margin'].sum():>12,.0f} €")
print(f"  Taux marge moy : {df['margin'].sum()/df['revenue'].sum()*100:>11.1f} %")
print(f"  Revenus négatifs : {(df['revenue'] < 0).sum()} (attendu : 0)")

### 4.3 — Agrégats client (base de la segmentation RFM)

> **MÉTHODE — Pourquoi `nunique()` pour `nb_orders` et non `count()` ?**
>
> Un client avec 3 produits dans 1 commande a 3 lignes dans notre dataset. `count()` retournerait 3. `nunique()` retourne 1 (1 commande unique). La **fréquence** dans RFM compte les commandes, pas les lignes produits.
>
> **Date de référence statique (`'2024-12-31'`)** plutôt que `pd.Timestamp.now()`. Pourquoi ? Si on utilise 'aujourd'hui', le même notebook produit des résultats différents à 3 mois d'intervalle. Avec une date fixe, les résultats sont reproductibles et auditables — essentiel en data engineering professionnel.

In [ ]:
# Ancienneté client
TODAY = pd.Timestamp("2024-12-31")
customers["customer_age_days"] = (TODAY - customers["date_inscription"]).dt.days

# KPIs par client (sur toutes les commandes, pas seulement livrées)
client_kpis = (
    df.groupby("customer_id")
    .agg(
        nb_orders     = ("order_id",  "nunique"),   # Fréquence RFM
        total_revenue = ("revenue",   "sum"),        # Monetary RFM
        total_margin  = ("margin",    "sum"),
        avg_basket    = ("montant_total", "mean"),   # Panier moyen
    )
    .reset_index()
)

print(f"KPIs calculés pour {len(client_kpis):,} clients")
print(f"  nb_orders moyen        : {client_kpis['nb_orders'].mean():.1f} commandes")
print(f"  total_revenue médian   : {client_kpis['total_revenue'].median():.0f} €")
print(f"  avg_basket moyen       : {client_kpis['avg_basket'].mean():.0f} €")
print(f"  Clients 1 seule commande : {(client_kpis['nb_orders']==1).sum():,} ({(client_kpis['nb_orders']==1).mean()*100:.1f}%)")

> **MÉTIER — Le taux de clients à 1 seule commande.**
>
> Si X% des clients n'ont commandé qu'une fois, ils représentent Y€ de CA potentiel si on les convertit en acheteurs récurrents. Un programme de re-engagement ciblé (email J+30 post-achat) a un ROI estimé bien supérieur à une campagne générique.

### 4.4 — Flags métier et enrichissement

In [ ]:
df = (
    df
    .merge(client_kpis, on="customer_id", how="left")
    .merge(customers[["customer_id", "customer_age_days"]], on="customer_id", how="left")
)

# Flag VIP : CA total >= 1 000 €
# Justification : seuil à valider avec M. Diallo selon la distribution réelle
df["is_vip"] = (df["total_revenue"] >= 1000).astype(int)

# Flag commande à risque : délai > 10 jours ET pas encore livrée
# Justification : risque de générer un avis négatif ou une demande de remboursement
df["is_risky_order"] = (
    (df["delivery_delay"] > 10) &
    (df["order_status"] != "Livree")
).astype(int)

print(f"Dataset enrichi : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")
print(f"  Clients VIP (is_vip=1)   : {df[df['is_vip']==1]['customer_id'].nunique():,} clients uniques")
print(f"  Commandes à risque       : {df['is_risky_order'].sum():,} lignes")

> **MÉTHODE — `is_vip` au niveau ligne (et non client) : choix délibéré.**
>
> Dans le dataset analytique, 1 ligne = 1 produit dans 1 commande. Si un client VIP commande 3 produits, ses 3 lignes ont `is_vip = 1`. Pourquoi ? Pour faciliter les filtres SQL et DAX : `WHERE is_vip = 1` retourne toutes les lignes des clients VIP, permettant d'analyser quels produits ils achètent, quelles catégories ils préfèrent.
>
> Pour compter les clients VIP uniques : `COUNT(DISTINCT customer_id WHERE is_vip = 1)`

---
## 5. Dataset final — `fact_ecommerce_analytics`

> **MÉTHODE — Sélectionner les colonnes finales explicitement.**
>
> Après plusieurs merges et calculs, le dataset contient de nombreuses colonnes. Sélectionner explicitement les colonnes finales a trois avantages : dataset plus lisible et plus léger, intentions documentées, collisions de noms résolues.

In [ ]:
final_cols = [
    # Clés de jointure
    "order_id", "customer_id", "product_id",
    # Dimensions temporelles et commerciales
    "order_date", "mois", "canal", "payment_method", "order_status", "delivery_delay",
    # Mesures quantité / prix
    "quantite", "prix_unitaire", "cout_produit",
    # Variables financières calculées
    "revenue", "cost", "margin",
    # Dimensions produit
    "nom_produit", "categorie", "marque",
    # Dimensions client
    "segment_client", "pays", "ville",
    # Agrégats client
    "nb_orders", "total_revenue", "total_margin", "avg_basket", "customer_age_days",
    # Flags métier
    "is_vip", "is_risky_order",
]

final_cols  = [c for c in final_cols if c in df.columns]
fact_ec     = df[final_cols].copy()

print(f"Colonnes finales  : {len(final_cols)}")
print(f"Lignes finales    : {len(fact_ec):,}")
print(f"Nulls restants    : {fact_ec.isnull().sum().sum()}")
print(f"Taille en mémoire : {fact_ec.memory_usage(deep=True).sum() / 1024**2:.1f} MB")


fact_ec.head(3)

---
## 6. EDA post-nettoyage

> **MÉTHODE — Pourquoi une EDA après le nettoyage et non avant ?**
>
> L'EDA de l'étape 1 travaillait sur les données brutes. Les chiffres pouvaient être faussés par les doublons, les montants négatifs et les nulls. Cette EDA produit les premiers KPIs **fiables** que M. Diallo peut présenter au comité de direction.
>
> On compare systématiquement les résultats étape 1 vs étape 2 pour mesurer l'impact du nettoyage. Si les chiffres changent > 5%, le nettoyage a corrigé des anomalies significatives — toujours utiliser les chiffres post-nettoyage en restitution.

In [ ]:
df_liv = fact_ec[fact_ec["order_status"] == "Livree"].copy()

ca_total     = df_liv["revenue"].sum()
marge_totale = df_liv["margin"].sum()
nb_cmd       = df_liv["order_id"].nunique()
nb_clients   = df_liv["customer_id"].nunique()
panier_moyen = df_liv.groupby("order_id")["revenue"].sum().mean()
taux_marge   = marge_totale / ca_total * 100

print("=" * 55)
print("  KPIs GLOBAUX — ShopAfrica+ (post-nettoyage)")
print("=" * 55)
print(f"  CA total          : {ca_total:>12,.0f} €")
print(f"  Marge totale      : {marge_totale:>12,.0f} €")
print(f"  Taux de marge     : {taux_marge:>11.1f} %")
print(f"  Nb commandes      : {nb_cmd:>12,}")
print(f"  Nb clients actifs : {nb_clients:>12,}")
print(f"  Panier moyen      : {panier_moyen:>12.2f} €")
print("=" * 55)

In [ ]:
df_liv_m = df_liv.copy()
df_liv_m["mois"] = df_liv_m["order_date"].dt.to_period("M")
monthly = (
    df_liv_m.groupby("mois")
    .agg(ca=("revenue", "sum"), marge=("margin", "sum"))
    .reset_index()
    .assign(mois_str=lambda x: x["mois"].astype(str),
            taux_marge=lambda x: x["marge"] / x["ca"] * 100)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
idx = range(len(monthly))
z   = np.polyfit(list(idx), monthly["ca"].values, 1)

axes[0].bar(list(idx), monthly["ca"] / 1000, color=COLORS["primary"], alpha=0.75, edgecolor="white")
axes[0].plot(list(idx), np.poly1d(z)(list(idx)) / 1000,
             color=COLORS["danger"], linewidth=2, linestyle="--",
             label=f"Tendance ({z[0]/1000:+.1f}k€/mois)")
axes[0].set_xticks(list(idx)[::3])
axes[0].set_xticklabels(monthly["mois_str"].iloc[::3], rotation=45, ha="right", fontsize=9)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}k"))
axes[0].set_title("CA mensuel (post-nettoyage)", fontweight="bold")
axes[0].legend(fontsize=9)

axes[1].plot(list(idx), monthly["taux_marge"],
             color=COLORS["secondary"], linewidth=2.5, marker="o", markersize=4)
axes[1].axhline(monthly["taux_marge"].mean(), color=COLORS["neutral"], linestyle="--",
                linewidth=1.5, label=f"Moy. {monthly['taux_marge'].mean():.1f}%")
axes[1].set_xticks(list(idx)[::3])
axes[1].set_xticklabels(monthly["mois_str"].iloc[::3], rotation=45, ha="right", fontsize=9)
axes[1].set_title("Taux de marge mensuel (%)", fontweight="bold")
axes[1].legend(fontsize=9)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))

plt.tight_layout()
plt.show()
print(f"Tendance CA                : {z[0]/1000:+.1f}k€/mois")
print(f"Variabilité taux de marge  : {monthly['taux_marge'].std():.1f} pts (écart-type)")

> **INTERPRÉTATION — Évolution CA et taux de marge**
>
> **Une tendance CA négative + taux de marge stable ou en hausse** n'est pas forcément une mauvaise nouvelle — c'est le scénario d'une stratégie de premiumisation : on vend moins mais plus rentablement.
>
> **Si le taux de marge varie fortement (écart-type > 3 pts)**, identifier quels événements (soldes, campagnes) correspondent aux creux de marge.

In [ ]:
cat_perf = (
    df_liv.groupby("categorie")
    .agg(ca=("revenue", "sum"), marge=("margin", "sum"), qte=("quantite", "sum"))
    .assign(taux_marge=lambda x: x["marge"] / x["ca"] * 100)
    .sort_values("ca", ascending=False).reset_index()
)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors_cat = [COLORS["primary"] if i < 3 else COLORS["primary"] + "99" for i in range(len(cat_perf))]
axes[0].barh(cat_perf["categorie"], cat_perf["ca"] / 1000, color=colors_cat, alpha=0.85, edgecolor="white")
axes[0].set_title("CA par catégorie (k€)", fontweight="bold")
axes[0].invert_yaxis()

colors_m = [COLORS["secondary"] if v >= 35 else COLORS["warning"] if v >= 25 else COLORS["danger"] for v in cat_perf["taux_marge"]]
axes[1].barh(cat_perf["categorie"], cat_perf["taux_marge"], color=colors_m, alpha=0.85, edgecolor="white")
axes[1].axvline(cat_perf["taux_marge"].mean(), color=COLORS["neutral"], linestyle="--",
                linewidth=1.5, label=f"Moy. {cat_perf['taux_marge'].mean():.1f}%")
axes[1].set_title("Taux de marge par catégorie (%)", fontweight="bold")
axes[1].invert_yaxis()
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

display(cat_perf[["categorie", "ca", "marge", "taux_marge", "qte"]]
        .rename(columns={"ca": "CA (€)", "marge": "Marge (€)", "taux_marge": "Taux marge (%)", "qte": "Quantité"}))

In [ ]:
seg = (
    df_liv.groupby("segment_client")
    .agg(ca=("revenue", "sum"), marge=("margin", "sum"),
         nb_clients=("customer_id", "nunique"), panier=("avg_basket", "mean"))
    .assign(ca_par_client=lambda x: x["ca"] / x["nb_clients"],
            taux_marge=lambda x: x["marge"] / x["ca"] * 100)
    .sort_values("ca", ascending=False).reset_index()
)
print("Performance par segment client :")
display(seg)

ca_vip     = df_liv[df_liv["is_vip"] == 1]["revenue"].sum()
n_vip_uniq = df_liv[df_liv["is_vip"] == 1]["customer_id"].nunique()
n_tot_uniq = df_liv["customer_id"].nunique()
print(f"\nConcentration VIP :")
print(f"  {n_vip_uniq} clients VIP ({n_vip_uniq/n_tot_uniq*100:.1f}% des clients)")
print(f"  génèrent {ca_vip/ca_total*100:.1f}% du CA total")
print(f"  Ratio de concentration : {(ca_vip/ca_total)/(n_vip_uniq/n_tot_uniq):.1f}x")

> **INTERPRÉTATION — Concentration VIP**
>
> Si le ratio de concentration est > 3, c'est une confirmation du principe de Pareto (80/20). Perdre 10% des clients VIP impacte plus le CA que perdre 30% des clients Standard. Une stratégie de rétention dédiée aux VIP a un ROI bien supérieur à une rétention générique.

In [ ]:
rev_cat = (
    reviews
    .merge(products[["product_id", "categorie"]], on="product_id", how="left")
    .groupby("categorie")["rating"]
    .agg(note_moy="mean", nb_avis="count")
    .sort_values("note_moy").reset_index()
)

fig, ax = plt.subplots(figsize=(10, 5))
colors_r = [COLORS["danger"] if v < 3.5 else COLORS["warning"] if v < 4.0 else COLORS["secondary"] for v in rev_cat["note_moy"]]
bars = ax.barh(rev_cat["categorie"], rev_cat["note_moy"], color=colors_r, alpha=0.85, edgecolor="white")
for bar, v in zip(bars, rev_cat["note_moy"]):
    ax.text(v + 0.02, bar.get_y() + bar.get_height() / 2, f"{v:.2f}", va="center", fontsize=9)
ax.axvline(4.0, color=COLORS["neutral"], linestyle="--", linewidth=1.5, label="Cible 4/5")
ax.set_xlabel("Note moyenne / 5")
ax.set_title("Satisfaction client par catégorie", fontweight="bold")
ax.set_xlim(0, 5.2)
ax.legend()
plt.tight_layout()
plt.show()

print("\nCroisement CA vs Satisfaction :")
cross = cat_perf.merge(rev_cat, on="categorie", how="left")
for _, r in cross.iterrows():
    note   = f"{r['note_moy']:.2f}" if pd.notna(r["note_moy"]) else "N/A"
    alerte = " ⚠️ BOMBE À RETARDEMENT" if r["taux_marge"] > 30 and pd.notna(r["note_moy"]) and r["note_moy"] < 3.5 else ""
    print(f"  {r['categorie']:<25} | CA {r['ca']/1000:.0f}k€ | Marge {r['taux_marge']:.0f}% | Note {note}{alerte}")

> **INTERPRÉTATION — Le tableau croisé CA / Marge / Note**
>
> **Forte marge + mauvaise note (< 3.5) : 'BOMBE À RETARDEMENT'** → La catégorie est rentable aujourd'hui mais les avis négatifs vont décourager les futurs acheteurs. Le CA va baisser dans 3-6 mois si on ne corrige pas la cause.
>
> **Faible marge + bonne note** → Vérifier si ces clients achètent aussi des produits à forte marge (cross-sell). La catégorie peut jouer un rôle d'appel.

---
## 7. Export des fichiers

> Si travail en local, enregistrer dans ./outputs/
>
> sinon dans le Drive pour Colab

In [ ]:
# Export du dataset analytique principal
# Convertir mois (Period) en string pour la compatibilité CSV
fact_ec_export = fact_ec.copy()
fact_ec_export["mois"] = fact_ec_export["mois"].astype(str)

# Si travail en local, enregistrer dans ./outputs/ ; sinon dans le Drive pour Colab
fact_ec_export.to_csv(f"{SAVE_PATH}fact_ecommerce_analytics.csv", index=False)
print(f"✅ fact_ecommerce_analytics.csv exporté ({len(fact_ec_export):,} lignes)")
print(f"   Colonnes : {fact_ec_export.columns.tolist()}")

In [ ]:
# Export des tables dim & fact en CSV

# DIM_CUSTOMERS
dim_customers = customers[["customer_id", "pays", "ville", "segment_client",
                            "date_inscription", "customer_age_days"]].copy()
dim_customers.to_csv(f"{SAVE_PATH}dim_customers.csv", index=False)


# DIM_PRODUCTS
dim_products = products[["product_id", "nom_produit", "categorie", "marque",
                          "prix_unitaire", "cout_produit"]].copy()
dim_products.to_csv(f"{SAVE_PATH}dim_products.csv", index=False)

# DIM_DATE
all_dates = pd.date_range(start=orders["order_date"].min(),
                          end=orders["order_date"].max(), freq="D")
dim_date = pd.DataFrame({"date": all_dates})
dim_date["annee"]       = dim_date["date"].dt.year
dim_date["mois"]        = dim_date["date"].dt.month
dim_date["nom_mois"]    = dim_date["date"].dt.strftime("%B")
dim_date["trimestre"]   = dim_date["date"].dt.quarter
dim_date["jour_sem"]    = dim_date["date"].dt.dayofweek
dim_date["est_weekend"] = (dim_date["jour_sem"] >= 5).astype(int)
dim_date["date_id"]     = dim_date["date"].dt.strftime("%Y%m%d").astype(int)
dim_date.to_csv(f"{SAVE_PATH}dim_date.csv", index=False)

# FACT_ORDERS
fact_orders = orders[["order_id", "customer_id", "order_date", "canal",
                       "payment_method", "order_status", "montant_total", "delivery_delay"]].copy()
fact_orders["date_id"] = fact_orders["order_date"].dt.strftime("%Y%m%d").astype(int)
fact_orders.to_csv(f"{SAVE_PATH}fact_orders.csv", index=False)

# FACT_ORDER_ITEMS
fact_oi = order_items.copy()
fact_oi["line_revenue"] = fact_oi["quantite"] * fact_oi["prix_unitaire"]
cost_map = products.set_index("product_id")["cout_produit"]
fact_oi["line_cost"]    = fact_oi["quantite"] * fact_oi["product_id"].map(cost_map)
fact_oi["line_margin"]  = fact_oi["line_revenue"] - fact_oi["line_cost"]
fact_oi.to_csv(f"{SAVE_PATH}fact_order_items.csv", index=False)

reviews.to_csv(f"{SAVE_PATH}fact_reviews.csv",   index=False)
web_logs.to_csv(f"{SAVE_PATH}fact_web_logs.csv", index=False)

print("Fichiers exportés ✅")
print("  fact_ecommerce_analytics.csv  → utilisé dans l'étape 3 (DuckDB + RFM)")
print("  dim_customers.csv, dim_products.csv, dim_date.csv")
print("  fact_orders.csv, fact_order_items.csv, fact_reviews.csv, fact_web_logs.csv")

### comment recuperer les fichiers dans colab

Dans Colab, les fichiers sont sauvegardés dans votre Google Drive sous "DataProjectLab/projects/ecommerce_analytics/fact_ecommerce_analytics.csv".

Vous pouvez y accéder directement depuis votre Drive ou utiliser le code suivant pour le télécharger localement :

> from google.colab import files  
>
> files.download(f"{SAVE_PATH}fact_ecommerce_analytics.csv")

**Note** : le téléchargement direct peut ne pas fonctionner pour les fichiers volumineux (>100MB) en raison des limitations de Colab. Dans ce cas, il est recommandé de télécharger manuellement depuis le Drive.

Si vous travaillez en local, le fichier sera dans le dossier "outputs" de votre projet.

N'oubliez pas de vérifier que le chemin d'accès est correct selon votre environnement (Colab vs local) et que vous avez les permissions nécessaires pour accéder au fichier.

En cas de problème d'accès, assurez-vous que le fichier a bien été créé et que le chemin est correct. Vous pouvez aussi vérifier les logs de Colab pour confirmer l'emplacement d'enregistrement.

---
## 8. Synthèse

> **MÉTHODE — Les 4 principes que ce notebook applique.**
>
> **Principe 1 — Toujours justifier chaque décision de nettoyage.** Médiane pour `delivery_delay` car distribution asymétrique. Exclusion pour `montant_total` négatif car non représentable en e-commerce standard.
>
> **Principe 2 — Distinguer anomalie technique et anomalie métier.** Un délai négatif est techniquement impossible = corriger. Un panier à 5 000€ est inhabituel mais possible = signaler et conserver.
>
> **Principe 3 — Penser réutilisabilité du dataset final.** `fact_ecommerce_analytics` sert pour SQL (étape 3), Power BI (étape 4). Nommer explicitement, exporter en CSV ET en SQL, documenter les colonnes.
>
> **Principe 4 — L'assert comme filet de sécurité.** Un merge mal configuré produit des doublons silencieux. L'assert vérifie l'invariant immédiatement.

### Checklist de complétion

- [x] Données rechargées depuis les CSV bruts (autonomie du notebook)
- [x] Audit types : colonnes datetime vérifiées
- [x] Audit nulls : 3 catégories (légitime / technique / critique) documentées
- [x] Audit doublons : sur clé primaire et sur ligne complète
- [x] Audit cohérence : FK, montants négatifs, statuts normalisés
- [x] Nettoyage : imputation médiane, exclusion montants négatifs, normalisation statuts
- [x] Feature engineering : revenue, cost, margin, agrégats client, flags métier
- [x] Dataset final : colonnes documentées par groupe, export CSV
- [x] EDA post-nettoyage : KPIs, tendances, catégories, segments, satisfaction
- [x] Export SQLite : toutes les tables dim & fact

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.